In [1]:
import os
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional

import torch
from torch.utils.data import IterableDataset
from datasets import load_dataset, interleave_datasets
from transformers import Trainer, TrainingArguments, set_seed

import sys
sys.path.insert(0, "../")

from src.model.gigachat_vl import GigaChatVL
from src.dataset.finevision import load_finevision_streaming, \
    FineVisionIterableDataset, VLMDataCollator
from src.dataset.unified_vlm_dataset import load_merged_dataset, SupportedDatasets
from src.utils.train_utils import SaveVLMArtifactsCallback, save_artifacts
from src.model.smollm3_vl import SmolLM3VL

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu130 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [2]:
PROJECT_ROOT = "/media/alexey/SSDData/experiments/gigachat_vl/"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "gigachat_vl")

for p in [PROJECT_ROOT]:
    if p not in sys.path:
        sys.path.append(p)

os.makedirs(OUTPUT_DIR, exist_ok=True)

VISION_PATH = None

dataset_specs = [
    {
        "config": SupportedDatasets.LLAVA_PRETRAIN_RU.value,
        "limit": 100_000,
        "dataset_root": "/media/alexey/HDDLargeData/datasets/llm/VL/Maya/", 
        "visual_encoder": VISION_PATH,
    },
    {
        "config": SupportedDatasets.MSCOCO_CAPTION_ML.value,
        "limit": 30_000,
        "dataset_root": "/media/alexey/HDDLargeData/datasets/llm/VL/mscoco-multilingual-30k/",
        "visual_encoder": VISION_PATH,
    },
    {
        "config": SupportedDatasets.RUSTITW_OCR.value,
        "limit": 80_000,
        "dataset_root": "/media/alexey/HDDLargeData/datasets/llm/VL/rustitw_ocr/",
        "visual_encoder": VISION_PATH,
    },
    {
        "config": SupportedDatasets.OPENHERMES_RU_TEXT.value,
        "limit": 5_000,
        "dataset_root": "/media/alexey/HDDLargeData/datasets/llm/OpenHermes-2.5-ru/",
    },
    {
        "config": SupportedDatasets.GQA_RU.value,
        "limit": 50_000,
        "dataset_root": "/media/alexey/HDDLargeData/datasets/llm/VL/GRA-ru/",
        "visual_encoder": VISION_PATH,
    },
    # {
    #     "config": SupportedDatasets.SMOLTALK2_SFT.value,
    #     "dataset_root": "/media/alexey/HDDLargeData/datasets/llm/smoltalk2/",
    #     "limit": 5_000,
    #     "load_kwargs": {
    #         "splits": ["OpenHermes-2.5_no_think"],
    #     },
    #     "dataset_kwargs": {
    #         "strip_thinking": True,
    #         "emit_all_assistant_turns": True,
    #     },
    # },
]

In [3]:
# Path config to fix TypeError

# LLM_NAME = "/media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16"

# config_path = os.path.join(LLM_NAME, "config.json")

# with open(config_path, "r", encoding="utf-8") as f:
#     cfg = json.load(f)

# print("Before:", cfg.get("routed_scaling_factor"), type(cfg.get("routed_scaling_factor")))

# if "routed_scaling_factor" in cfg and isinstance(cfg["routed_scaling_factor"], int):
#     cfg["routed_scaling_factor"] = float(cfg["routed_scaling_factor"])

# with open(config_path, "w", encoding="utf-8") as f:
#     json.dump(cfg, f, ensure_ascii=False, indent=2)

# print("After:", cfg.get("routed_scaling_factor"), type(cfg.get("routed_scaling_factor")))
# print("Patched:", config_path)

In [ ]:
LLM_PATH = "/media/alexey/HDDLargeData/models/LLM/GigaChat3.1-10B-A1.8B-bf16"
# LLM_PATH = "/media/alexey/HDDLargeData/models/LLM/SmolLM3-3B/"
VISION_PATH = "/media/alexey/HDDLargeData/models/VL/siglip2-base-patch16-512/"
# VISION_PATH = "/media/alexey/HDDLargeData/models/VLM/Qwen2.5-VL-7B-Instruct/"
QWEN_PATH = "/media/alexey/HDDLargeData/models/LLM/Qwen3-0.6B/"

TRAIN_LLM_LORA = True
MAX_STEPS = 25_000
LR = 1e-5
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
warmup_steps = int(MAX_STEPS * WARMUP_RATIO)

TRAIN_BS = 2
GRAD_ACCUM = 16
MAX_LENGTH = 4096

SHUFFLE_BUFFER = 1000
LOGGING_STEPS = 25
SAVE_STEPS = 1000
SEED = 42

USE_4BIT_LLM = True
FREEZE_VISION = True
USE_PRECOMPUTED_VISION_EMBEDDINGS = False
REQUIRE_PRECOMPUTED_VISION_EMBEDDINGS = USE_PRECOMPUTED_VISION_EMBEDDINGS
DROP_FROZEN_VISION_AFTER_PROJECTOR = FREEZE_VISION and USE_PRECOMPUTED_VISION_EMBEDDINGS
PROJECTOR_PATH = None

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

NUM_LORA_LAYERS = -1

SHUFFLE_CONVERSATIONS = False
SKIP_MULTI_IMAGE = True
MAX_TURNS_PER_ROW = None

set_seed(SEED)

if USE_PRECOMPUTED_VISION_EMBEDDINGS and not FREEZE_VISION:
    raise ValueError("Precomputed vision embeddings can only be used with FREEZE_VISION=True.")

image_dataset_names = {
    SupportedDatasets.LLAVA_PRETRAIN_RU.value.name,
    SupportedDatasets.MSCOCO_CAPTION_ML.value.name,
    SupportedDatasets.RUSTITW_OCR.value.name,
    SupportedDatasets.GQA_RU.value.name,
    SupportedDatasets.LLAVA_INSTRUCT_RU.value.name,
    SupportedDatasets.MWS_VISION.value.name,
    SupportedDatasets.PIXMO_CAP_EN.value.name,
    SupportedDatasets.PIXMO_ASK_MODEL_ANYTHING_EN.value.name,
    SupportedDatasets.DOCVQA_EN.value.name,
    SupportedDatasets.INFOGRAPHICVQA_EN.value.name,
    SupportedDatasets.CHARTQA_EN.value.name,
}
if not TRAIN_LLM_LORA:
    dataset_specs = [
        spec for spec in dataset_specs
        if spec["config"].name in image_dataset_names
    ]

for spec in dataset_specs:
    if spec["config"].name in image_dataset_names:
        if USE_PRECOMPUTED_VISION_EMBEDDINGS:
            spec["visual_encoder"] = VISION_PATH
        else:
            spec.pop("visual_encoder", None)

model = GigaChatVL(
    llm_name=LLM_PATH,
    vision_name=VISION_PATH,
    use_4bit_llm=USE_4BIT_LLM,
    freeze_vision=FREEZE_VISION,
    projector_type="mlp",
    # projector_path=PROJECTOR_PATH,
    # projector_type="llm",
    # connector_llm_name=QWEN_PATH,
    # freeze_connector_llm=False,
    # connector_llm_use_qlora=True,
    # connector_lora_r=16,
    # connector_lora_alpha=32,
    # connector_lora_dropout=0.05,
    # vision_use_lora=True,
    # vision_use_qlora=False,
    # vision_lora_r=16,
    # vision_lora_alpha=32,
    # vision_lora_dropout=0.05,
    # lora_r=LORA_R,
    # lora_alpha=LORA_ALPHA,
    # lora_dropout=LORA_DROPOUT,
    num_lora_layers=NUM_LORA_LAYERS,
    train_llm_lora=TRAIN_LLM_LORA,
    
    normalize_visual_embeddings=True,
    enable_vl_experts=False,
    vl_expert_layers=[],
)

# model = SmolLM3VL(
#     llm_name=LLM_PATH,
#     vision_name=VISION_PATH,
#     use_4bit_llm=True,
#     freeze_vision=True,
#     lora_r=LORA_R,
#     lora_alpha=LORA_ALPHA,
#     lora_dropout=LORA_DROPOUT,
#     num_lora_layers=NUM_LORA_LAYERS,
#     train_llm_lora=True,
#     projector_type="mlp",
#     # projector_type="llm",
#     # connector_llm_name=QWEN_PATH,
#     # freeze_connector_llm=False,
#     # connector_llm_use_qlora=True,
#     # connector_lora_r=16,
#     # connector_lora_alpha=32,
#     # connector_lora_dropout=0.05,
# )

if hasattr(model.llm, "gradient_checkpointing_enable"):
    try:
        model.llm.gradient_checkpointing_enable()
    except Exception as e:
        print(f"gradient_checkpointing_enable failed: {e}")

if hasattr(model.llm, "print_trainable_parameters"):
    try:
        model.llm.print_trainable_parameters()
    except Exception as e:
        print(f"print_trainable_parameters failed: {e}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"VLM trainable params: {trainable_params:,} / {all_params:,} ({100 * trainable_params / all_params:.4f}%)")


train_dataset = load_merged_dataset(
    dataset_specs=dataset_specs,
    global_seed=SEED,
    global_shuffle_buffer=SHUFFLE_BUFFER,
    interleave_stopping_strategy="all_exhausted",
    interleave_balance_probabilities=False,
)

if DROP_FROZEN_VISION_AFTER_PROJECTOR:
    model.drop_frozen_vision_modules()

collator = VLMDataCollator(
    model=model,
    max_length=MAX_LENGTH,
    require_precomputed_vision_embeddings=REQUIRE_PRECOMPUTED_VISION_EMBEDDINGS,
)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    optim="paged_adamw_8bit",
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=TRAIN_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    bf16=use_bf16,
    fp16=use_fp16,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
    callbacks=[SaveVLMArtifactsCallback()],
)

model.gradient_checkpointing_enable()

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Some weights of the model checkpoint at /media/alexey/HDDLargeData/models/LLM/GigaChat3.1-10B-A1.8B-bf16 were not used when initializing DeepseekV3ForCausalLM: ['model.layers.26.eh_proj.weight', 'model.layers.26.embed_tokens.weight', 'model.layers.26.enorm.weight', 'model.layers.26.hnorm.weight', 'model.layers.26.input_layernorm.weight', 'model.layers.26.mlp.experts.0.down_proj.weight', 'model.layers.26.mlp.experts.0.gate_proj.weight', 'model.layers.26.mlp.experts.0.up_proj.weight', 'model.layers.26.mlp.experts.1.down_proj.weight', 'model.layers.26.mlp.experts.1.gate_proj.weight', 'model.layers.26.mlp.experts.1.up_proj.weight', 'model.layers.26.mlp.experts.10.down_proj.weight', 'model.layers.26.mlp.experts.10.gate_proj.weight', 'model.layers.26.mlp.experts.10.up_proj.weight', 'model.layers.26.mlp.experts.11.down_proj.weight', 'model.layers.26.mlp.experts.11.gate_proj.weight', 'model.layers.26.mlp.experts.11.up_proj.weight', 'model.layers.26.mlp.experts.12.down_proj.weight', 'model.laye

Note: base LLM checkpoint contains an extra MTP block (210 tensors under `model.layers.26.*`). Transformers `DeepseekV3ForCausalLM` does not load this auxiliary MTP block for standard causal LM training/inference, so these unexpected keys are expected and can be ignored.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
trainer.train(
    # resume_from_checkpoint="/media/alexey/SSDData/experiments/gigachat_vl/outputs/gigachat_vl_siglipv2/checkpoint-5000/"
)

Step,Training Loss
25,2.820700


KeyboardInterrupt: 

In [6]:
save_artifacts(model, OUTPUT_DIR)
print(f"Saved final artifacts to: {OUTPUT_DIR}")

/home/alexey/venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Saved final artifacts to: /media/alexey/SSDData/experiments/gigachat_vl/outputs/gigachat_vl


/home/alexey/venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(
